# Airline AI Assistant

## Importing the libraries

In [2]:
import os
import json
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

## Loading OpenAI API Key

In [9]:
load_dotenv(override = True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key found!")
else:
    print("API key found!")

API key found!


## Getting started with tools

In [4]:
ticket_prices = {
    "london": "$799",
    "paris": "$899",
    "tokyo": "1400",
    "berlin": "$499"
}

In [5]:
def get_ticket_prices(city):
    print(f"Tool called for the city {city}")
    price = ticket_prices.get(city.lower(), "unknown")
    return f"The price of a ticket to {city} is {price}."

In [6]:
ticket_prices_desc = {
    "name": "get_ticket_prices",
    "description": "Get the price of a return ticket to the destination city!",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city that the customer wants to travel to"
            }
        }
    },
    "required": ["city"],
    "additionalProperties": False
}

In [ ]:
tools = [{
    "type": "function",
    "function": ticket_prices_desc
}]

## Calling OpenAI with tools

In [10]:
openai = OpenAI()

In [11]:
system_prompt = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [30]:
def handle_tool_calls(message):

    responses = []
    for tool_call in message.tool_calls:
        
        if tool_call.function.name == "get_ticket_prices":

            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("city")
            price_details = get_ticket_prices(city)

            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })

    return responses

In [ ]:
def chat(message, history):

    history = [{
        "role": h["role"],
        "content": h["content"]
    } for h in history ]

    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(
        model = "gpt-5.4-mini",
        messages = messages,
        tools = tools
    )

    while response.choices[0].finish_reason == "tool_calls":

        message = response.choices[0].message
        tool_response = handle_tool_calls(message)
        messages.append(message)
        messages.extend(tool_response)
        response = openai.chat.completions.create(
            model = "gpt-5.4-mini",
            messages = messages
        )
    
    return response.choices[0].message.content

## Creating a chat interface

In [35]:
interface = gr.ChatInterface(
    fn = chat
)

In [36]:
interface.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Tool called for the city Berlin
Tool called for the city Paris


In [37]:
interface.close()

Closing server running on port: 7863


## Creating a SQL database

In [38]:
import sqlite3

In [72]:
DB = "ticket_prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)")
    conn.commit()

In [73]:
def get_ticket_prices_db(city):

    print(f"Database tool called! Getting prices for {city}", flush = True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT price FROM prices WHERE city = ?", (city.lower(), ))
        result = cursor.fetchone()
        return f"The price of a ticket to {city} is ${result[0]}." if result else "No price data is available for this city"

In [74]:
def set_ticket_prices_db(city, price):
    
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?",
            (city.lower(), price, price)
        )
        conn.commit()

In [75]:
ticket_prices_db = {
    "london": 799,
    "paris": 899,
    "tokyo": 1420,
    "sydney": 2999
}

In [76]:
for city, price in ticket_prices_db.items():
    set_ticket_prices_db(city, price)

In [77]:
def handle_tool_calls_db(message):

    responses = []
    for tool_call in message.tool_calls:
        
        if tool_call.function.name == "get_ticket_prices":

            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("city")
            price_details = get_ticket_prices_db(city)

            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })

    return responses

In [78]:
def chat_db(message, history):

    history = [{
        "role": h["role"],
        "content": h["content"]
    } for h in history ]

    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(
        model = "gpt-5.4-mini",
        messages = messages,
        tools = tools
    )

    while response.choices[0].finish_reason == "tool_calls":

        message = response.choices[0].message
        tool_response = handle_tool_calls_db(message)
        messages.append(message)
        messages.extend(tool_response)
        response = openai.chat.completions.create(
            model = "gpt-5.4-mini",
            messages = messages
        )
    
    return response.choices[0].message.content

In [79]:
interface = gr.ChatInterface(
    fn = chat_db
)

In [ ]:
interface.launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


Database tool called! Getting prices for Sydney
Database tool called! Getting prices for Paris


In [81]:
interface.close()

Closing server running on port: 7868
